# 04.1 — PMI & Co-occurrence Matrices

**Problem it solves:** Represent word meaning as a vector. If 'bank' and 'river' co-occur often, they're related. PMI quantifies how much more often two words co-occur than chance.

**PMI(w, c) = log P(w,c) / P(w)P(c)**

This is the mathematical foundation of Word2Vec, GloVe, and all distributional semantics.

---

In [ ]:
import numpy as np
import re
from collections import Counter
import math

def tokenize(text): return re.findall(r'\b[a-z]+\b', text.lower())

def build_cooccurrence_matrix(corpus: list[str], window: int = 2) -> tuple:
    """
    Build a word x context co-occurrence matrix.
    For each word, count how many times each other word
    appears within `window` positions.
    """
    # Build vocab from corpus
    all_tokens = []
    for sent in corpus:
        all_tokens.extend(tokenize(sent))
    
    word_counts = Counter(all_tokens)
    vocab = sorted(word_counts.keys())
    word2idx = {w: i for i, w in enumerate(vocab)}
    V = len(vocab)
    
    # Build co-occurrence matrix
    cooc = np.zeros((V, V), dtype=float)
    
    for sent in corpus:
        tokens = tokenize(sent)
        for i, word in enumerate(tokens):
            if word not in word2idx:
                continue
            w_idx = word2idx[word]
            # Context window
            for j in range(max(0, i-window), min(len(tokens), i+window+1)):
                if i == j:
                    continue
                context = tokens[j]
                if context not in word2idx:
                    continue
                c_idx = word2idx[context]
                # Distance-weighted (GloVe style)
                distance = abs(i - j)
                cooc[w_idx][c_idx] += 1.0 / distance
    
    return cooc, vocab, word2idx

corpus = [
    "the cat sat on the mat",
    "the cat ate the mouse",
    "the dog chased the cat",
    "the dog ran in the park",
    "dogs and cats are pets",
    "the bank is near the river",
    "the river bank has water",
    "the financial bank lends money",
    "money in the bank earns interest",
    "machine learning processes data",
    "deep learning uses neural networks",
]

cooc, vocab, word2idx = build_cooccurrence_matrix(corpus, window=2)
print(f"Vocab size: {len(vocab)}")
print(f"Co-occurrence matrix: {cooc.shape}")

# Show co-occurrences for 'bank'
bank_idx = word2idx.get('bank')
if bank_idx is not None:
    bank_row = cooc[bank_idx]
    top = sorted(enumerate(bank_row), key=lambda x: x[1], reverse=True)[:10]
    print("\nTop co-occurrences for 'bank':")
    for idx, count in top:
        if count > 0:
            print(f"  {vocab[idx]:15} {count:.2f}")

In [ ]:
def compute_ppmi(cooc: np.ndarray, k: float = 1.0) -> np.ndarray:
    """
    Positive PMI (PPMI): PMI clipped at 0.
    Negative PMI values are unreliable (sparse data), so we discard them.
    
    PMI(w,c) = log[ P(w,c) / (P(w) * P(c)) ]
    PPMI(w,c) = max(0, PMI(w,c))
    
    k: context distribution smoothing (Levy et al. 2015)
       smoothing context counts by raising to power alpha (usually 0.75)
    """
    # Total count
    total = cooc.sum()
    
    # Marginal probabilities
    p_w = cooc.sum(axis=1) / total  # P(word)
    p_c = cooc.sum(axis=0) / total  # P(context)
    
    # Joint probability
    p_wc = cooc / total
    
    # PMI with numerical stability
    eps = 1e-10
    # Broadcasting: p_w (V,) * p_c (V,) -> outer product (V,V)
    denominator = np.outer(p_w, p_c)
    
    # Avoid log(0)
    ratio = np.divide(p_wc, denominator + eps, 
                      where=(denominator > eps))
    ratio = np.maximum(ratio, eps)
    
    pmi = np.log2(ratio + eps)
    
    # PPMI: clip at 0
    ppmi = np.maximum(pmi, 0)
    
    return ppmi

ppmi = compute_ppmi(cooc)

# Now PPMI vectors are 'word embeddings'
# Reduce dimensionality with SVD for dense representations
U, S, Vt = np.linalg.svd(ppmi, full_matrices=False)
k = min(10, len(vocab) - 1)
word_vectors = U[:, :k] * S[:k]  # truncated SVD

print(f"Word vectors shape: {word_vectors.shape}")
print(f"(Each word is now a {k}-dimensional vector)")

In [ ]:
def cosine_sim(v1: np.ndarray, v2: np.ndarray) -> float:
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0: return 0.0
    return float(np.dot(v1, v2) / (n1 * n2))

def most_similar(word: str, vectors: np.ndarray, vocab: list, 
                 word2idx: dict, n: int = 5) -> list:
    if word not in word2idx:
        return []
    idx = word2idx[word]
    vec = vectors[idx]
    sims = [(cosine_sim(vec, vectors[i]), vocab[i]) 
            for i in range(len(vocab)) if i != idx]
    return sorted(sims, reverse=True)[:n]

print("Similarity from PPMI + SVD word vectors:")
for word in ['cat', 'dog', 'bank', 'learning']:
    if word in word2idx:
        similar = most_similar(word, word_vectors, vocab, word2idx)
        print(f"\n  Similar to '{word}':")
        for sim, w in similar:
            print(f"    {w:15} {sim:.3f}")

In [ ]:
# PMI is the theoretical backbone of Word2Vec
# Levy & Goldberg (2014) proved:
# Word2Vec skip-gram with negative sampling implicitly factorizes
# a shifted PMI matrix: PMI(w,c) - log(k)
# where k = number of negative samples

print("PMI connection to Word2Vec (Levy & Goldberg 2014):")
print()
print("Skip-gram with k negative samples optimizes:")
print("  w · c - log k  (for each positive pair)")
print()
print("This is exactly PPMI with a shift of log(k).")
print("Understanding PMI deeply = understanding what Word2Vec actually learns.")
print()
print("Intuition:")
print("  High PMI(bank, river) -> 'bank' and 'river' are contextually related")
print("  High PMI(bank, money) -> 'bank' (financial) and 'money' are related")
print("  Ambiguous word? PMI vector is superposition of both senses.")

## Summary

**Distributional hypothesis:** Words with similar contexts have similar meanings (Harris, 1954). This is the theoretical foundation of all modern NLP.

**PPMI + SVD** is essentially the predecessor of Word2Vec. It works reasonably well but requires O(V²) memory — impractical for large vocabularies.

**What breaks it:**
- Polysemy: 'bank' has one vector even though it has multiple meanings
- Memory: V=100k words → 10B matrix entries
- Context window size is a hyperparameter that significantly affects results
- Rare words have noisy co-occurrence statistics

---
**Next:** 04.2 — Word2Vec Skip-gram from scratch